# 🏗️ Module 06 — CI/CD Pipelines for AI/ML
## Ship Models Like Software

> **Marevlo AI Platform** · All Levels: Beginner → Expert

---
### What you'll learn
- How ML CI/CD differs from software CI/CD (data + model + code versioning)
- DVC pipeline definitions and data versioning with S3
- MLflow experiment tracking, model registration, and stage promotion
- GitHub Actions workflows for training, evaluation, and deployment
- Docker multi-stage builds for lean inference containers
- Hard evaluation gates that block deployment on metric regression
- Canary deployments and automated rollback on SLA breach
- Data drift detection and shadow mode evaluation

---

In [ ]:
!pip install mlflow dvc scikit-learn xgboost pydantic fastapi prometheus-client scipy pandas numpy -q

---
## 🟢 Part 1 — DVC Pipeline & Data Versioning
### Beginner: DVC Concepts

In [ ]:
# DVC Core Concepts — illustrated without requiring a remote

import subprocess, os, tempfile, json
from pathlib import Path

# Show what a .dvc pointer file looks like
DVC_POINTER = """
# data/raw/telemetry.parquet.dvc
# This tiny file is committed to Git
# It points to the actual data stored in S3

outs:
- md5: a3f8c2b14d9e7f6c1a2b3c4d5e6f7890
  size: 124857392
  path: telemetry.parquet
"""
print("DVC pointer file (.dvc):")
print(DVC_POINTER)

# Show what dvc.yaml pipeline definition looks like
DVC_PIPELINE = """
# dvc.yaml — defines the reproducible ML pipeline
stages:
  ingest:
    cmd: python src/ingest.py
    deps: [src/ingest.py, configs/params.yaml]
    outs: [data/raw/telemetry.parquet]

  features:
    cmd: python src/features.py
    deps: [src/features.py, data/raw/telemetry.parquet]
    outs: [data/features/X_train.npy, data/features/y_train.npy]

  train:
    cmd: python src/train.py
    deps: [src/train.py, data/features/X_train.npy]
    params: [configs/params.yaml: [model.xgb_n_estimators, model.xgb_max_depth]]
    outs: [models/hunter_xgb.pkl]
    metrics: [metrics/train_metrics.json]

  evaluate:
    cmd: python src/evaluate.py
    deps: [src/evaluate.py, models/hunter_xgb.pkl]
    metrics: [metrics/eval_metrics.json]
"""
print("DVC pipeline definition (dvc.yaml):")
print(DVC_PIPELINE)

In [ ]:
import yaml

# Params file — single source of truth for ALL hyperparameters
PARAMS = {
    "ingest": {
        "lookback_days": 7,
        "min_fpc_coverage": 0.6,
        "s3_bucket": "marevlo-hunter-data"
    },
    "features": {
        "window_sizes": [1, 6, 24],
        "null_threshold": 0.3,
        "delta_features": True
    },
    "model": {
        "xgb_n_estimators": 200,
        "xgb_max_depth": 6,
        "xgb_learning_rate": 0.1,
        "iso_contamination": 0.05,
        "anomaly_threshold": 0.75,
        "random_seed": 42
    },
    "evaluate": {
        "min_precision": 0.85,
        "min_recall": 0.80,
        "min_auc": 0.90,
        "max_regression": 0.02
    }
}

print("configs/params.yaml (single source of truth):")
print(yaml.dump(PARAMS, default_flow_style=False))

### 🟡 Intermediate: Data Validation Gate

In [ ]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
import sys

@dataclass
class DataValidationResult:
    passed:   bool
    errors:   list
    warnings: list
    stats:    dict

def validate_telemetry(df: pd.DataFrame) -> DataValidationResult:
    """Validate Hunter telemetry dataset. Exits with code 1 on hard failure in CI."""
    errors, warnings = [], []

    # ── Schema checks ────────────────────────────────────
    required_cols = ["lc_id", "fpc_id", "ts", "cpu_util", "mem_util"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        errors.append(f"Missing required columns: {missing}")

    # ── Size checks ──────────────────────────────────────
    if len(df) < 100:  # lowered for demo
        errors.append(f"Too few rows: {len(df)} (min: 100)")
    elif len(df) < 500:
        warnings.append(f"Low row count: {len(df)} (recommended: 500+)")

    # ── FPC coverage check ───────────────────────────────
    fpc_coverage = 0.0
    if "fpc_id" in df.columns:
        fpc_coverage = df["fpc_id"].notna().mean()
        if fpc_coverage < 0.6:
            errors.append(f"FPC coverage too low: {fpc_coverage:.1%} (min: 60%)")

    # ── Null rate checks ─────────────────────────────────
    null_rates = df.isnull().mean()
    high_null = null_rates[null_rates > 0.3]
    if not high_null.empty:
        warnings.append(f"High null rate: {high_null.to_dict()}")

    # ── Value range checks ───────────────────────────────
    if "cpu_util" in df.columns:
        if df["cpu_util"].max() > 100 or df["cpu_util"].min() < 0:
            errors.append("cpu_util values outside [0, 100]")

    stats = {
        "rows": len(df),
        "cols": len(df.columns),
        "fpc_coverage": float(fpc_coverage),
        "null_rate": float(df.isnull().mean().mean())
    }

    for e in errors:   print(f"  ✗ ERROR: {e}")
    for w in warnings: print(f"  ⚠ WARN:  {w}")

    if not errors:
        print(f"  ✓ Validation PASSED ({len(df):,} rows, {len(df.columns)} cols)")

    return DataValidationResult(passed=not errors, errors=errors, warnings=warnings, stats=stats)

# Test with a synthetic dataset
np.random.seed(42)
n = 500

# GOOD dataset
good_df = pd.DataFrame({
    "lc_id":    [f"LC-{i:04d}" for i in range(n)],
    "fpc_id":   [f"FPC-{i//4:04d}" for i in range(n)],
    "ts":       pd.date_range("2024-01-01", periods=n, freq="h"),
    "cpu_util": np.random.uniform(20, 95, n),
    "mem_util": np.random.uniform(30, 85, n),
})

# BAD dataset (missing columns, too few rows)
bad_df = pd.DataFrame({
    "device":   ["x"] * 50,  # Wrong column name
    "cpu_util": np.random.uniform(0, 100, 50),
})

print("\n=== GOOD DATASET ===")
good_result = validate_telemetry(good_df)

print("\n=== BAD DATASET ===")
bad_result = validate_telemetry(bad_df)

print(f"\nGood: passed={good_result.passed}, stats={good_result.stats}")
print(f"Bad:  passed={bad_result.passed}, errors={bad_result.errors}")

---
## 🟡 Part 2 — MLflow Experiment Tracking
### Intermediate: Log a Training Run

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import numpy as np, json
from pathlib import Path

# Use a local MLflow tracking server (file-based for demo)
mlflow.set_tracking_uri("mlruns")  # Saves to ./mlruns/
mlflow.set_experiment("hunter-v2")

# Generate synthetic Hunter-like dataset
np.random.seed(42)
n = 2000
X = np.column_stack([
    np.random.uniform(20, 100, n),   # cpu_util
    np.random.uniform(30, 90, n),    # mem_util
    np.random.exponential(20, n),    # pfe_errors
    np.random.randint(0, 12, n),     # bgp_flaps
])
# Label: high CPU + high errors = anomaly
y = ((X[:, 0] > 80) & (X[:, 2] > 100)).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model hyperparams from our params.yaml equivalent
model_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "random_state": 42
}

with mlflow.start_run(run_name="hunter-xgb-demo") as run:
    # ── Log all hyperparameters ──────────────────────
    mlflow.log_params(model_params)
    mlflow.log_param("train_samples", len(X_train))
    mlflow.log_param("features", ["cpu_util", "mem_util", "pfe_errors", "bgp_flaps"])

    # ── Train ────────────────────────────────────────
    model = GradientBoostingClassifier(**model_params)
    model.fit(X_train, y_train)

    # ── Evaluate ─────────────────────────────────────
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_test, y_pred, zero_division=0)),
        "auc":       float(roc_auc_score(y_test, y_proba))
    }

    # ── Log metrics ──────────────────────────────────
    mlflow.log_metrics(metrics)

    # ── Save metrics to file (for DVC / CI check) ────
    Path("metrics").mkdir(exist_ok=True)
    Path("metrics/eval_metrics.json").write_text(json.dumps(metrics, indent=2))
    mlflow.log_artifact("metrics/eval_metrics.json")

    # ── Log model ────────────────────────────────────
    mlflow.sklearn.log_model(model, "hunter_xgb")

    run_id = run.info.run_id
    print(f"\n✓ MLflow Run: {run_id}")
    print(f"  Precision: {metrics['precision']:.3f}")
    print(f"  Recall:    {metrics['recall']:.3f}")
    print(f"  AUC:       {metrics['auc']:.3f}")
    print(f"\n  View at: mlflow ui --port 5000")

### 🔴 Advanced: Hard Evaluation Gates

In [ ]:
import json, sys
from pathlib import Path

THRESHOLDS = {
    "precision": 0.85,
    "recall":    0.80,
    "auc":       0.90,
}

def check_gates(metrics: dict, thresholds: dict = THRESHOLDS, exit_on_fail: bool = False) -> bool:
    """Hard evaluation gates — in CI this exits with code 1 on failure."""
    passed_gates, failed_gates = [], []

    for metric, threshold in thresholds.items():
        value = metrics.get(metric)
        if value is None:
            failed_gates.append(f"  ✗ {metric}: MISSING")
        elif value < threshold:
            failed_gates.append(f"  ✗ {metric}: {value:.3f} < {threshold:.3f}  [FAILED]")
        else:
            passed_gates.append(f"  ✓ {metric}: {value:.3f} >= {threshold:.3f}")

    print("=== Evaluation Gate Results ===")
    for msg in passed_gates + failed_gates:
        print(msg)

    if failed_gates:
        print(f"\n✗ {len(failed_gates)} gate(s) FAILED — deployment blocked")
        if exit_on_fail:
            sys.exit(1)  # Fails the CI step
        return False

    print(f"\n✓ All {len(passed_gates)} gates PASSED — proceeding to deployment")
    return True

# Scenario 1: Model that passes all gates
good_metrics = {"precision": 0.891, "recall": 0.847, "auc": 0.935}
print("\nSCENARIO 1: Passing model")
result1 = check_gates(good_metrics)

# Scenario 2: Model that fails precision gate
bad_metrics = {"precision": 0.831, "recall": 0.812, "auc": 0.905}
print("\nSCENARIO 2: Failing model (precision below threshold)")
result2 = check_gates(bad_metrics)

# Scenario 3: Model that fails multiple gates
very_bad_metrics = {"precision": 0.721, "recall": 0.643, "auc": 0.781}
print("\nSCENARIO 3: Model with multiple failures")
result3 = check_gates(very_bad_metrics)

print(f"\nResults: {result1=}, {result2=}, {result3=}")

### 🟣 Expert: Champion vs Challenger Comparison

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("mlruns")
client = MlflowClient()

def compare_with_production(
    challenger_metrics: dict,
    production_metrics: dict,
    max_regression: float = 0.02
) -> tuple[bool, list[str]]:
    """
    Fail CI if challenger is significantly worse than production.
    Returns (passed, messages)
    """
    issues, info = [], []

    for metric in ["precision", "recall", "auc"]:
        prod_val = production_metrics.get(metric, 0.0)
        chal_val = challenger_metrics.get(metric, 0.0)
        delta    = chal_val - prod_val

        if delta < -max_regression:
            issues.append(f"  ✗ {metric} regressed: {prod_val:.3f} → {chal_val:.3f} (delta={delta:+.3f}, limit={-max_regression:.3f})")
        elif delta >= 0:
            info.append(f"  ✓ {metric} improved:  {prod_val:.3f} → {chal_val:.3f} ({delta:+.3f})")
        else:
            info.append(f"  ~ {metric} slight drop: {prod_val:.3f} → {chal_val:.3f} ({delta:+.3f}) [within tolerance]")

    return (len(issues) == 0), (info + issues)

# Simulate current production model
production_metrics = {"precision": 0.872, "recall": 0.834, "auc": 0.921}

# Challenger: improved model
challenger_v1 = {"precision": 0.891, "recall": 0.847, "auc": 0.935}
# Challenger: slightly regressed model
challenger_v2 = {"precision": 0.851, "recall": 0.810, "auc": 0.903}
# Challenger: significantly regressed model
challenger_v3 = {"precision": 0.841, "recall": 0.800, "auc": 0.895}

print("Production baseline:", production_metrics)

for name, challenger in [("v1 (improved)", challenger_v1), ("v2 (slight drop)", challenger_v2), ("v3 (regression)", challenger_v3)]:
    passed, messages = compare_with_production(challenger, production_metrics)
    print(f"\n{'✓ PASS' if passed else '✗ FAIL'} — Challenger {name}:")
    for msg in messages:
        print(msg)

---
## 🔴 Part 3 — Data Drift Detection

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

def detect_distribution_drift(
    reference: pd.DataFrame,
    current:   pd.DataFrame,
    columns:   list[str],
    threshold: float = 0.05  # KS p-value threshold
) -> dict:
    """Detect feature distribution drift using Kolmogorov-Smirnov test."""
    results = {}
    drifted = []

    for col in columns:
        if col not in reference.columns or col not in current.columns:
            continue

        ref_vals = reference[col].dropna()
        cur_vals = current[col].dropna()

        ks_stat, p_value = stats.ks_2samp(ref_vals, cur_vals)
        drifted_flag = p_value < threshold

        results[col] = {
            "ks_statistic": round(ks_stat, 4),
            "p_value":      round(p_value, 4),
            "drifted":      drifted_flag,
            "ref_mean":     round(float(ref_vals.mean()), 3),
            "cur_mean":     round(float(cur_vals.mean()), 3),
            "delta_mean":   round(float(cur_vals.mean() - ref_vals.mean()), 3)
        }
        if drifted_flag:
            drifted.append(col)

    drift_rate = len(drifted) / max(len(columns), 1)

    print("=== Data Drift Detection Results ===")
    for col, r in results.items():
        status = "⚠️  DRIFT" if r["drifted"] else "✓  OK"
        print(f"  {status} {col}: p={r['p_value']:.4f}, mean: {r['ref_mean']} → {r['cur_mean']} ({r['delta_mean']:+.3f})")

    print(f"\nDrift rate: {drift_rate:.0%} ({len(drifted)}/{len(columns)} features)")
    if drift_rate > 0.3:
        print("✗ Drift rate exceeds 30% threshold — retrain required!")
    else:
        print("✓ Drift within acceptable limits")

    return {"drift_rate": drift_rate, "drifted": drifted, "details": results}

np.random.seed(42)
n = 1000

# Reference distribution (training time)
reference = pd.DataFrame({
    "cpu_util":   np.random.normal(55, 15, n),
    "mem_util":   np.random.normal(60, 12, n),
    "pfe_errors": np.random.exponential(20, n),
    "bgp_flaps":  np.random.poisson(2, n),
})

# Current production distribution — CPU has drifted significantly
current_drifted = pd.DataFrame({
    "cpu_util":   np.random.normal(78, 18, n),  # ← Drifted! Mean 55 → 78
    "mem_util":   np.random.normal(62, 13, n),  # ← OK
    "pfe_errors": np.random.exponential(22, n), # ← Slight change, likely OK
    "bgp_flaps":  np.random.poisson(2, n),      # ← OK
})

print("\n=== SCENARIO: CPU Drift ===")
result = detect_distribution_drift(
    reference, current_drifted,
    columns=["cpu_util", "mem_util", "pfe_errors", "bgp_flaps"]
)

---
## 🟢 Part 4 — Inference Server Design
### FastAPI + MLflow: Production Inference API

In [ ]:
# Inference server structure (conceptual — run separately with uvicorn)

SERVER_CODE = '''
# src/server.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Literal
import mlflow, numpy as np, os, time
from prometheus_client import Counter, Histogram, generate_latest

app = FastAPI(title="Hunter Inference API", version="2.0")

# Prometheus metrics
REQUESTS = Counter("hunter_requests_total", "Total requests", ["risk_level"])
LATENCY  = Histogram("hunter_latency_seconds", "Latency in seconds")

model = None  # Loaded at startup

class InferenceRequest(BaseModel):
    device_id:  str = Field(pattern=r"^[A-Z]{2,4}-\\d{3}$")
    cpu_util:   float = Field(ge=0, le=100)
    mem_util:   float = Field(ge=0, le=100)
    pfe_errors: int   = Field(ge=0)
    bgp_flaps:  int   = Field(ge=0)

class InferenceResponse(BaseModel):
    device_id:     str
    risk_level:    Literal["LOW", "MED", "HIGH"]
    score:         float
    model_version: str

@app.on_event("startup")
async def load_model():
    global model
    model_uri = os.getenv("MODEL_URI", "models:/hunter-xgb-v2/Production")
    model = mlflow.sklearn.load_model(model_uri)

@app.post("/predict", response_model=InferenceResponse)
async def predict(req: InferenceRequest):
    t0 = time.perf_counter()
    X  = np.array([[req.cpu_util, req.mem_util, req.pfe_errors, req.bgp_flaps]])
    p  = float(model.predict_proba(X)[0, 1])
    risk = "HIGH" if p >= 0.8 else ("MED" if p >= 0.5 else "LOW")

    REQUESTS.labels(risk_level=risk).inc()
    LATENCY.observe(time.perf_counter() - t0)

    return InferenceResponse(device_id=req.device_id, risk_level=risk,
                             score=round(p, 3), model_version="2.0")

@app.get("/health")
async def health():
    if model is None:
        raise HTTPException(503, "Model not loaded")
    return {"status": "healthy", "model": "hunter-xgb-v2"}

@app.get("/metrics")
async def prom_metrics():
    return generate_latest()

# Run with: uvicorn src.server:app --host 0.0.0.0 --port 8080
'''
print("Inference server code (src/server.py):")
print(SERVER_CODE)

In [ ]:
# Simulate the inference logic without a running server
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier

# Use the model we trained earlier
# (re-train quickly for demo)
np.random.seed(42)
n = 2000
X = np.column_stack([
    np.random.uniform(20, 100, n),
    np.random.uniform(30, 90, n),
    np.random.exponential(20, n),
    np.random.randint(0, 12, n),
])
y = ((X[:, 0] > 80) & (X[:, 2] > 100)).astype(int)
demo_model = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X, y)

def predict(model, device_id, cpu_util, mem_util, pfe_errors, bgp_flaps):
    X = np.array([[cpu_util, mem_util, pfe_errors, bgp_flaps]])
    p = float(model.predict_proba(X)[0, 1])
    risk = "HIGH" if p >= 0.8 else ("MED" if p >= 0.5 else "LOW")
    return {"device_id": device_id, "risk_level": risk, "score": round(p, 3)}

# Test cases
test_cases = [
    ("JNP-001", 94.2, 67.3, 412, 8),   # Should be HIGH
    ("JNP-002", 45.0, 52.1, 3,   0),   # Should be LOW
    ("JNP-003", 78.3, 61.0, 87,  3),   # Should be MED-ish
]

print("Inference results:")
for args in test_cases:
    result = predict(demo_model, *args)
    print(f"  {result['device_id']}: {result['risk_level']} (score={result['score']:.3f})")

---
## 🟣 Part 5 — GitHub Actions Workflow Design
### Expert: Complete Workflow Analysis

In [ ]:
# Complete workflow structure for Hunter ML pipeline
# This shows the dependency graph and expected run times

WORKFLOW_SPEC = {
    "name": "Hunter Complete ML Pipeline",
    "triggers": ["push:main", "schedule:weekly", "workflow_dispatch"],
    "jobs": [
        {
            "id": "code-quality",
            "needs": [],
            "runner": "ubuntu-latest",
            "steps": ["checkout", "lint (ruff)", "type-check (mypy)", "unit-tests (pytest)"],
            "expected_minutes": 3,
            "fail_blocks": "all"
        },
        {
            "id": "train-evaluate",
            "needs": ["code-quality"],
            "runner": "ubuntu-latest",
            "steps": ["dvc pull", "validate data", "dvc repro", "check gates", "register model"],
            "expected_minutes": 25,
            "fail_blocks": "all downstream"
        },
        {
            "id": "build-container",
            "needs": ["train-evaluate"],
            "runner": "ubuntu-latest",
            "steps": ["docker build", "trivy scan", "push to ECR", "sign image"],
            "expected_minutes": 8,
            "fail_blocks": "deploy"
        },
        {
            "id": "deploy-staging",
            "needs": ["build-container"],
            "runner": "ubuntu-latest",
            "environment": "staging",
            "steps": ["deploy to staging", "smoke tests", "integration tests"],
            "expected_minutes": 10,
            "fail_blocks": "production deploy"
        },
        {
            "id": "deploy-production",
            "needs": ["deploy-staging"],
            "runner": "ubuntu-latest",
            "environment": "production (requires approval)",
            "steps": ["canary 10%", "validate (10min)", "canary 50%", "validate (10min)", "canary 100%", "promote model"],
            "expected_minutes": 30,
            "rollback": "automatic on failure"
        }
    ]
}

print(f"Workflow: {WORKFLOW_SPEC['name']}")
print(f"Triggers: {', '.join(WORKFLOW_SPEC['triggers'])}")
print()

total_minutes = sum(j["expected_minutes"] for j in WORKFLOW_SPEC["jobs"])
for job in WORKFLOW_SPEC["jobs"]:
    deps = f" (needs: {', '.join(job['needs'])})" if job["needs"] else " (start)"
    env  = f" [{job['environment']}]" if "environment" in job else ""
    print(f"  {job['id']}{deps}{env}")
    print(f"    Steps: {' → '.join(job['steps'])}")
    print(f"    Time:  ~{job['expected_minutes']}min")
    if "rollback" in job:
        print(f"    Rollback: {job['rollback']}")
    print()

print(f"Total sequential time: ~{total_minutes}min")

---
## 🏆 Part 6 — Canary Deployment Simulation

In [ ]:
import time, random
from dataclasses import dataclass, field

@dataclass
class CanarySimulator:
    """Simulates canary deployment with health monitoring and auto-rollback."""
    
    # SLA thresholds
    max_error_rate:   float = 0.01   # 1% error rate
    max_p95_latency:  float = 0.500  # 500ms
    max_high_risk_rt: float = 0.40   # 40% HIGH risk predictions

    # State
    canary_weight:  int   = 0
    deployment_log: list  = field(default_factory=list)
    rolled_back:    bool  = False
    
    def _get_health(self, healthy: bool = True) -> dict:
        """Simulate health metrics. Unhealthy = simulate SLA breach."""
        if healthy:
            return {
                "error_rate":   random.uniform(0.001, 0.008),
                "p95_latency":  random.uniform(0.120, 0.380),
                "high_risk_rt": random.uniform(0.12, 0.28),
            }
        else:
            # Simulate a bad deploy
            return {
                "error_rate":   random.uniform(0.015, 0.040),  # > 1%!
                "p95_latency":  random.uniform(0.600, 1.200),  # > 500ms!
                "high_risk_rt": random.uniform(0.35, 0.55),
            }
    
    def _check_sla(self, health: dict) -> list[str]:
        breaches = []
        if health["error_rate"]   > self.max_error_rate:   breaches.append(f"error_rate={health['error_rate']:.3f} > {self.max_error_rate}")
        if health["p95_latency"]  > self.max_p95_latency:  breaches.append(f"p95={health['p95_latency']:.3f}s > {self.max_p95_latency}s")
        if health["high_risk_rt"] > self.max_high_risk_rt: breaches.append(f"high_risk={health['high_risk_rt']:.2%} > {self.max_high_risk_rt:.0%}")
        return breaches
    
    def deploy(self, healthy: bool = True):
        steps = [10, 50, 100]
        
        for weight in steps:
            print(f"\n→ Deploying canary at {weight}% traffic...")
            self.canary_weight = weight
            self.deployment_log.append(f"canary:{weight}%")
            
            # Simulate validation window
            health = self._get_health(healthy)
            breaches = self._check_sla(health)
            
            print(f"  error_rate:   {health['error_rate']:.3f} {'⚠' if health['error_rate'] > self.max_error_rate else '✓'}")
            print(f"  p95_latency:  {health['p95_latency']:.3f}s {'⚠' if health['p95_latency'] > self.max_p95_latency else '✓'}")
            print(f"  high_risk_rt: {health['high_risk_rt']:.2%} {'⚠' if health['high_risk_rt'] > self.max_high_risk_rt else '✓'}")
            
            if breaches:
                print(f"\n✗ SLA BREACH at {weight}%: {breaches}")
                print(f"  → Initiating automatic rollback...")
                self._rollback()
                return False
            else:
                print(f"  ✓ SLA OK at {weight}% — promoting")
        
        print(f"\n✓ Canary deployment complete at 100%")
        self.deployment_log.append("deployed:production")
        return True
    
    def _rollback(self):
        self.rolled_back = True
        self.canary_weight = 0
        self.deployment_log.append("rollback:previous_version")
        print(f"  ✓ Rolled back to previous production version")

# Scenario 1: Healthy deploy
print("=" * 50)
print("SCENARIO 1: Healthy Deployment")
print("=" * 50)
sim1 = CanarySimulator()
success1 = sim1.deploy(healthy=True)
print(f"\nDeployment log: {sim1.deployment_log}")

# Scenario 2: Unhealthy deploy (auto-rollback)
print("\n" + "=" * 50)
print("SCENARIO 2: Unhealthy Deploy → Auto-Rollback")
print("=" * 50)
random.seed(0)  # Make it deterministic for demo
sim2 = CanarySimulator()
success2 = sim2.deploy(healthy=False)
print(f"\nDeployment log: {sim2.deployment_log}")
print(f"Rolled back: {sim2.rolled_back}")

---
## 🏆 Module Challenge

Build a **complete mini ML CI/CD system** that:

1. **Data validation** — write a `validate_dataset(df)` function that checks:
   - Required columns present
   - No null rate > 40% in any column
   - At least 200 rows
   - All numeric columns within expected ranges

2. **Training + MLflow** — write a `train_and_log(X_train, y_train, params)` function that:
   - Trains a model with given params
   - Logs all params and metrics to MLflow
   - Returns (model, run_id, metrics)

3. **Evaluation gate** — write a `check_gates(metrics, thresholds)` function that:
   - Returns True if all metrics pass their thresholds
   - Raises ValueError with specific failure messages if any gate fails

4. **Champion/challenger** — write a `should_deploy(challenger_metrics, champion_metrics)` function that:
   - Returns True only if challenger passes gates AND doesn't regress >2% vs champion

5. **Compose into pipeline** — write a `run_pipeline(df, champion_metrics)` function that runs 1→4 in sequence, returning `{"deployed": bool, "reason": str, "metrics": dict}`.

**Bonus:** Add a `generate_ci_yaml()` function that prints a GitHub Actions workflow YAML for running your pipeline.

In [ ]:
# Your solution here!
import mlflow
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("hunter-challenge")

def validate_dataset(df: pd.DataFrame) -> tuple[bool, list[str]]:
    """Returns (passed, error_messages)."""
    # Your implementation here
    errors = []
    return len(errors) == 0, errors

def train_and_log(X_train, y_train, params: dict):
    """Returns (model, run_id, metrics)."""
    # Your implementation here
    pass

def check_gates(metrics: dict, thresholds: dict) -> bool:
    """Returns True if all metrics >= thresholds. Raises ValueError on failure."""
    # Your implementation here
    return True

def should_deploy(challenger: dict, champion: dict, max_regression: float = 0.02) -> tuple[bool, str]:
    """Returns (deploy, reason)."""
    # Your implementation here
    return True, "All checks passed"

def run_pipeline(df: pd.DataFrame, champion_metrics: dict) -> dict:
    """Full mini pipeline. Returns {deployed, reason, metrics}."""
    # Your implementation here
    pass

# Test your implementation with synthetic data
# np.random.seed(42)
# n = 500
# test_df = pd.DataFrame({
#     "cpu_util":   np.random.uniform(20, 100, n),
#     "mem_util":   np.random.uniform(30, 90, n),
#     "pfe_errors": np.random.exponential(20, n).astype(int),
#     "bgp_flaps":  np.random.randint(0, 12, n),
#     "label":      np.random.randint(0, 2, n)
# })
# champion = {"precision": 0.82, "recall": 0.79, "auc": 0.88}
# result = run_pipeline(test_df, champion)
# print(result)

---
## 📚 Course Summary — All 6 Modules

| Module | Topic | Key Tools |
|--------|--------|-----------|
| **01** | Data Formats | JSON, YAML, TOML, JSONL |
| **02** | JSON Schema | Pydantic, jsonschema, Instructor |
| **03** | XML & Markdown | Prompt structuring, injection defense |
| **04** | BAML & Pydantic | Type-safe LLM pipelines |
| **05** | Jinja2 | Dynamic prompt templates |
| **06** | CI/CD | GitHub Actions, DVC, MLflow, Docker |

### Module 06 Critical Rules
1. **Version everything**: code (Git) + data (DVC) + models (MLflow) — all three, independently
2. **Hard gates, not soft warnings**: `sys.exit(1)` on metric failure — never log-and-continue in CI
3. **Compare vs champion**: new models must not regress >2% on key metrics vs production
4. **Multi-stage Docker**: lean runtime images, no build toolchain in production containers
5. **Canary first**: 10% → 50% → 100%, with SLA validation at each step
6. **Automatic rollback**: `if: failure()` in GitHub Actions triggers rollback.py — no manual intervention
7. **Monitor for drift**: KS test on feature distributions in nightly CI — drift > 30% triggers retrain
8. **OIDC over long-lived keys**: use GitHub OIDC for keyless AWS auth in CI — no secrets rotation

---
**🎉 Course Complete — Agentic AI: Data Formats & DevOps**